# Pathfinding via Reinforcement and Imitation Multi-Agent Learning (PRIMAL) — Curriculum Version

While training is taking place, statistics on agent performance are available from Tensorboard. To launch it use:

`tensorboard --logdir train_primal`

This version adds **dynamic curriculum scheduling**: the environment difficulty (map size & obstacle density) automatically adjusts based on recent agent performance across 17 stages.


In [1]:
from __future__ import division

import os
USE_GPU = True
if not USE_GPU:
    os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_XLA_FLAGS'] = '--tf_xla_auto_jit=0 --tf_xla_enable_xla_devices=false'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['CUDA_MODULE_LOADING'] = 'LAZY'  # reduce initial GPU memory footprint
os.environ['MALLOC_ARENA_MAX'] = '2'  # reduce glibc arena count to prevent pthread_create failures

# Ensure the Jupyter kernel can find pip-installed NVIDIA libraries (cuDNN 8, etc.)
# This must happen BEFORE importing tensorflow.
import site, glob
_nvidia_base = os.path.join(site.getsitepackages()[0], 'nvidia')
if os.path.isdir(_nvidia_base):
    _nvidia_lib_dirs = glob.glob(os.path.join(_nvidia_base, '*', 'lib'))
    _ld = os.environ.get('LD_LIBRARY_PATH', '')
    for d in _nvidia_lib_dirs:
        if d not in _ld:
            _ld = d + ':' + _ld
    os.environ['LD_LIBRARY_PATH'] = _ld

import gym
import numpy as np
import random
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
import matplotlib.pyplot as plt
from od_mstar3 import cpp_mstar
from od_mstar3.col_set_addition import OutOfTimeError,NoSolutionError
import threading
# Reduce per-thread stack from 8MB to 2MB to prevent memory allocation failures
threading.stack_size(2 * 1024 * 1024)
import time
import scipy.signal as signal
import GroupLock
import multiprocessing
import gc
from tqdm.auto import tqdm as tqdm_bar
%matplotlib inline
import mapf_gym as mapf_gym
import pickle
import imageio
from ACNet import ACNet

# Do NOT probe for GPU here — it creates an extra CUDA context that wastes ~500MB VRAM.
# The training cell will detect GPU via allow_soft_placement.
print('USE_GPU =', USE_GPU)
print('TF version:', tf.__version__)
print('LD_LIBRARY_PATH:', os.environ.get('LD_LIBRARY_PATH', 'NOT SET')[:200])

2026-04-22 00:51:50.141058: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-22 00:51:50.159935: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-22 00:51:50.159970: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Instructions for updating:
non-resource variables are not supported in the long term
USE_GPU = True
TF version: 2.16.2
LD_LIBRARY_PATH: /opt/conda/envs/primal/lib/python3.10/site-packages/nvidia/cusolver/lib:/opt/conda/envs/primal/lib/python3.10/site-packages/nvidia/nccl/lib:/opt/conda/envs/primal/lib/python3.10/site-packages/nvidia/c


/opt/conda/envs/primal/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Helper Functions

In [2]:
def make_gif(images, fname, duration=2, true_image=False,salience=False,salIMGS=None):
    imageio.mimwrite(fname,images,subrectangles=True)
    print("wrote gif")

# Copies one set of variables to another.
# Used to set worker network parameters to those of global network.
def update_target_graph(from_scope,to_scope):
    from_vars = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, from_scope)
    to_vars = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, to_scope)

    op_holder = []
    for from_var,to_var in zip(from_vars,to_vars):
        op_holder.append(to_var.assign(from_var))
    return op_holder

def discount(x, gamma):
    x = np.asarray(x, dtype=np.float64)
    return signal.lfilter([1], [1, -gamma], x[::-1], axis=0)[::-1]

def good_discount(x, gamma):
    return discount(x,gamma)

## Curriculum Scheduler

In [3]:
class CurriculumScheduler:
    """
    Dynamic curriculum scheduler for PRIMAL training.

    Difficulty stages progress from small/sparse maps to large/dense maps.
    Every EVAL_WINDOW episodes (after WARMUP_EPISODES), upgrade/downgrade
    conditions are checked.  A stage change requires CONSEC_REQUIRED
    consecutive passing evaluations, after which a COOLDOWN_EPISODES
    observation period begins (no evaluation during cooldown).

    Stage table
    -----------
    0-5  : 10×10 map,  density 0.00-0.10 → 0.30-0.40
    6-11 : 15×15 map,  density 0.00-0.10 → 0.30-0.40
    12-16: 20×20 map,  density 0.00-0.10 → 0.20-0.30
    """

    STAGES = [
        {"size": (10, 10),  "density": (0.00, 0.10)},  # 0
        {"size": (10, 10),  "density": (0.05, 0.15)},  # 1
        {"size": (10, 10),  "density": (0.10, 0.20)},  # 2
        {"size": (10, 10),  "density": (0.15, 0.25)},  # 3
        {"size": (10, 10),  "density": (0.20, 0.30)},  # 4
        {"size": (10, 10),  "density": (0.30, 0.40)},  # 5
        {"size": (15, 15),  "density": (0.00, 0.10)},  # 6
        {"size": (15, 15),  "density": (0.05, 0.15)},  # 7
        {"size": (15, 15),  "density": (0.10, 0.20)},  # 8
        {"size": (15, 15),  "density": (0.15, 0.25)},  # 9
        {"size": (15, 15),  "density": (0.20, 0.30)},  # 10
        {"size": (15, 15),  "density": (0.30, 0.40)},  # 11
        {"size": (20, 20),  "density": (0.00, 0.10)},  # 12
        {"size": (20, 20),  "density": (0.05, 0.15)},  # 13
        {"size": (20, 20),  "density": (0.10, 0.20)},  # 14
        {"size": (20, 20),  "density": (0.15, 0.25)},  # 15
        {"size": (20, 20),  "density": (0.20, 0.30)},  # 16
    ]

    EVAL_WINDOW       = 200   # evaluate every N episodes
    WARMUP_EPISODES   = 2000  # no evaluation before this many episodes
    COOLDOWN_EPISODES = 200   # grace period (no evaluation) after a stage change
    CONSEC_REQUIRED   = 2     # consecutive passes needed to trigger a stage change

    # File name used to persist the curriculum stage alongside the model checkpoint
    STAGE_FILENAME    = 'curriculum_stage.txt'

    def __init__(self, initial_stage=0):
        self._lock           = threading.Lock()
        self.stage           = initial_stage
        self._last_eval_ep   = 0   # episode_count at which last evaluation ran
        self._cooldown_until = 0   # episode_count below which evaluation is suppressed
        self._up_streak      = 0   # consecutive upgrade evaluations
        self._down_streak    = 0   # consecutive downgrade evaluations
        self.gameEnvs        = None

    # ------------------------------------------------------------------
    @property
    def current_size(self):
        return self.STAGES[self.stage]["size"]

    @property
    def current_density(self):
        return self.STAGES[self.stage]["density"]

    # ------------------------------------------------------------------
    def save_stage(self, model_path):
        """Persist current stage to disk alongside the model checkpoint."""
        try:
            path = os.path.join(model_path, self.STAGE_FILENAME)
            with open(path, 'w') as f:
                f.write(str(self.stage))
        except Exception as e:
            print(f"[Curriculum] Warning: could not save stage: {e}")

    def load_stage(self, model_path):
        """
        Restore stage from disk.  Must be called BEFORE register_envs() so
        that envs are initialised with the correct size/density.
        Returns True if a saved stage was found.
        """
        path = os.path.join(model_path, self.STAGE_FILENAME)
        try:
            with open(path, 'r') as f:
                self.stage = int(f.read().strip())
            print(f"[Curriculum] Restored Stage {self.stage} "
                  f"(size={self.current_size}, density={self.current_density}) from {path}")
            return True
        except FileNotFoundError:
            print(f"[Curriculum] No saved stage found at {path}, starting from Stage {self.stage}")
            return False
        except Exception as e:
            print(f"[Curriculum] Warning: could not load stage: {e}")
            return False

    # ------------------------------------------------------------------
    def register_envs(self, envs):
        """
        Call this after gameEnvs are created to allow dynamic env updates.
        Immediately syncs all environments to the current stage.
        """
        self.gameEnvs = envs
        self._apply_stage(self.stage)  # ensure envs start at the correct stage

    # ------------------------------------------------------------------
    def _apply_stage(self, stage):
        """Push new SIZE/PROB into all registered game environments."""
        if self.gameEnvs is None:
            return
        new_size    = self.STAGES[stage]["size"]
        new_density = self.STAGES[stage]["density"]
        for env in self.gameEnvs:
            env.SIZE = new_size
            env.PROB = new_density

    # ------------------------------------------------------------------
    def maybe_evaluate(self, episode_count, mean_reward, mean_length,
                       valid_rate, mean_success, max_episode_length,
                       global_summary):
        """
        Called periodically to check whether the stage should change.

        Returns
        -------
        (changed: bool, new_stage: int)
        """
        with self._lock:
            # Respect EVAL_WINDOW: only evaluate once per window
            if episode_count - self._last_eval_ep < self.EVAL_WINDOW:
                return False, self.stage

            self._last_eval_ep = episode_count

            # Skip during warmup or post-change cooldown
            in_warmup   = episode_count < self.WARMUP_EPISODES
            in_cooldown = episode_count < self._cooldown_until
            if in_warmup or in_cooldown:
                # Still log success_rate and stage to TensorBoard
                if global_summary is not None:
                    summary = tf.Summary()
                    summary.value.add(tag='Curriculum/Stage',        simple_value=float(self.stage))
                    summary.value.add(tag='Curriculum/Success_Rate', simple_value=float(mean_success))
                    global_summary.add_summary(summary, int(episode_count))
                    global_summary.flush()
                return False, self.stage

            # ------ Upgrade condition (ALL must hold) ------
            upgrade_pass = (
                mean_success > 0.85 and
                mean_length  < 0.75 * max_episode_length and
                mean_reward  > -80 and
                valid_rate   > 0.995
            )
            # ------ Downgrade condition (ANY triggers) ------
            downgrade_pass = (
                mean_success < 0.60 or
                mean_length  > 0.92 * max_episode_length or
                valid_rate   < 0.985 or
                mean_reward  < -105
            )

            if upgrade_pass:
                self._up_streak   += 1
                self._down_streak  = 0
            elif downgrade_pass:
                self._down_streak += 1
                self._up_streak    = 0
            else:
                self._up_streak   = 0
                self._down_streak = 0

            old_stage = self.stage
            changed   = False

            if self._up_streak >= self.CONSEC_REQUIRED and self.stage < len(self.STAGES) - 1:
                self.stage += 1
                self._up_streak   = 0
                self._down_streak = 0
                self._cooldown_until = episode_count + self.COOLDOWN_EPISODES
                changed = True
                self._apply_stage(self.stage)
                print(f"\n[Curriculum] ▲ Upgraded  to Stage {self.stage:2d} "
                      f"(size={self.current_size}, density={self.current_density}) "
                      f"at ep {int(episode_count)}")

            elif self._down_streak >= self.CONSEC_REQUIRED and self.stage > 0:
                self.stage -= 1
                self._up_streak   = 0
                self._down_streak = 0
                self._cooldown_until = episode_count + self.COOLDOWN_EPISODES
                changed = True
                self._apply_stage(self.stage)
                print(f"\n[Curriculum] ▼ Downgraded to Stage {self.stage:2d} "
                      f"(size={self.current_size}, density={self.current_density}) "
                      f"at ep {int(episode_count)}")

            # Log to TensorBoard every evaluation (changed or not)
            if global_summary is not None:
                summary = tf.Summary()
                summary.value.add(tag='Curriculum/Stage',        simple_value=float(self.stage))
                summary.value.add(tag='Curriculum/Success_Rate', simple_value=float(mean_success))
                summary.value.add(tag='Curriculum/Up_Streak',    simple_value=float(self._up_streak))
                summary.value.add(tag='Curriculum/Down_Streak',  simple_value=float(self._down_streak))
                global_summary.add_summary(summary, int(episode_count))
                global_summary.flush()

            return changed, self.stage


## Worker

In [4]:
class Worker:
    def __init__(self, game, metaAgentID, workerID, a_size, groupLock):
        self.workerID = workerID
        self.env = game
        self.metaAgentID = metaAgentID
        self.name = "worker_"+str(workerID)
        # Agent index is always within [1, NUM_AGENTS] for this environment.
        self.agentID = ((workerID-1) % NUM_AGENTS) + 1 
        self.groupLock = groupLock

        self.nextGIF = episode_count # For GIFs output
        #Create the local copy of the network and the tensorflow op to copy global parameters to local network
        self.local_AC = ACNet(self.name,a_size,trainer,True,GRID_SIZE,GLOBAL_NET_SCOPE)
        self.pull_global = update_target_graph(GLOBAL_NET_SCOPE, self.name)

    def synchronize(self):
        #handy thing for keeping track of which to release and acquire
        if(not hasattr(self,"lock_bool")):
            self.lock_bool=False
        self.groupLock.release(int(self.lock_bool),self.name)
        self.groupLock.acquire(int(not self.lock_bool),self.name)
        self.lock_bool=not self.lock_bool
        
    def train(self, rollout, sess, gamma, bootstrap_value, rnn_state0, imitation=False):
        global episode_count
        if imitation:
            rollout=np.array(rollout, dtype=object)
            if rollout.ndim < 2 or rollout.shape[0] == 0:
                return 0.0
            #we calculate the loss differently for imitation
            #if imitation=True the rollout is assumed to have different dimensions:
            #[o[0],o[1],optimal_actions]
            feed_dict={global_step:episode_count,
                       self.local_AC.inputs:np.stack(rollout[:,0]),
                       self.local_AC.goal_pos:np.stack(rollout[:,1]),
                       self.local_AC.optimal_actions:np.stack(rollout[:,2]),
                       self.local_AC.state_in[0]:rnn_state0[0],
                       self.local_AC.state_in[1]:rnn_state0[1]
                      }
            _,i_l,_=sess.run([self.local_AC.policy,self.local_AC.imitation_loss,
                              self.local_AC.apply_imitation_grads],
                             feed_dict=feed_dict)
            return i_l
        rollout = np.array(rollout, dtype=object)
        if rollout.ndim < 2 or rollout.shape[0] == 0:
            return 0, 0, 0, 0, 0, 0, 0, 0
        observations = rollout[:,0]
        goals=rollout[:,-2]
        actions = rollout[:,1]
        rewards = rollout[:,2]
        values = rollout[:,5]
        valids = rollout[:,6]
        blockings = rollout[:,10]
        on_goals=rollout[:,8]
        train_value = rollout[:,-1]

        # Here we take the rewards and values from the rollout, and use them to 
        # generate the advantage and discounted returns. (With bootstrapping)
        # The advantage function uses "Generalized Advantage Estimation"
        self.rewards_plus = np.asarray(rewards.tolist() + [bootstrap_value])
        discounted_rewards = discount(self.rewards_plus,gamma)[:-1]
        self.value_plus = np.asarray(values.tolist() + [bootstrap_value])
        advantages = rewards + gamma * self.value_plus[1:] - self.value_plus[:-1]
        advantages = good_discount(advantages,gamma)

        num_samples = min(EPISODE_SAMPLES,len(advantages))
        sampleInd = np.sort(np.random.choice(advantages.shape[0], size=(num_samples,), replace=False))

        # Update the global network using gradients from loss
        # Generate network statistics to periodically save
        feed_dict = {
            global_step:episode_count,
            self.local_AC.target_v:np.stack(discounted_rewards),
            self.local_AC.inputs:np.stack(observations),
            self.local_AC.goal_pos:np.stack(goals),
            self.local_AC.actions:actions,
            self.local_AC.train_valid:np.stack(valids),
            self.local_AC.advantages:advantages,
            self.local_AC.train_value:train_value,
            self.local_AC.target_blockings:blockings,
            self.local_AC.target_on_goals:on_goals,
            self.local_AC.state_in[0]:rnn_state0[0],
            self.local_AC.state_in[1]:rnn_state0[1]
        }
        
        v_l,p_l,valid_l,e_l,g_n,v_n,b_l,og_l,_ = sess.run([self.local_AC.value_loss,
            self.local_AC.policy_loss,
            self.local_AC.valid_loss,
            self.local_AC.entropy,
            self.local_AC.grad_norms,
            self.local_AC.var_norms,
            self.local_AC.blocking_loss,
            self.local_AC.on_goal_loss,
            self.local_AC.apply_grads],
            feed_dict=feed_dict)
        return v_l/len(rollout), p_l/len(rollout), valid_l/len(rollout), e_l/len(rollout), b_l/len(rollout), og_l/len(rollout), g_n, v_n

    def shouldRun(self, coord, episode_count):
        if TRAINING:
            return (not coord.should_stop()) and (MAX_EPISODES == 0 or episode_count < MAX_EPISODES)
        else:
            return (episode_count < NUM_EXPS)

    def parse_path(self,path):
        '''needed function to take the path generated from M* and create the 
        observations and actions for the agent
        path: the exact path ouput by M*, assuming the correct number of agents
        returns: the list of rollouts for the "episode": 
                list of length num_agents with each sublist a list of tuples 
                (observation[0],observation[1],optimal_action,reward)'''
        result=[[] for i in range(NUM_AGENTS)]
        for t in range(len(path[:-1])):
            observations=[]
            move_queue=list(range(NUM_AGENTS))
            for agent in range(1,NUM_AGENTS+1):
                observations.append(self.env._observe(agent))
            steps=0
            while len(move_queue)>0:
                steps+=1
                i=move_queue.pop(0)
                o=observations[i]
                pos=path[t][i]
                newPos=path[t+1][i]#guaranteed to be in bounds by loop guard
                direction=(newPos[0]-pos[0],newPos[1]-pos[1])
                a=self.env.world.getAction(direction)
                state, reward, done, nextActions, on_goal, blocking, valid_action=self.env._step((i+1,a))
                if steps>NUM_AGENTS**2:
                    #if we have a very confusing situation where lots of agents move
                    #in a circle (difficult to parse and also (mostly) impossible to learn)
                    return None
                if not valid_action:
                    #the tie must be broken here
                    move_queue.append(i)
                    continue
                result[i].append([o[0],o[1],a])
        return result
        
    def work(self,max_episode_length,gamma,sess,coord,saver,pbar):
        global episode_count, swarm_reward, episode_rewards, episode_lengths, \
               episode_mean_values, episode_invalid_ops, episode_wrong_blocking, \
               episode_successes
        total_steps, i_buf = 0, 0
        episode_buffers, s1Values = [ [] for _ in range(NUM_BUFFERS) ], [ [] for _ in range(NUM_BUFFERS) ]

        try:
            with sess.as_default(), sess.graph.as_default():
                while self.shouldRun(coord, episode_count):
                    sess.run(self.pull_global)
    
                    episode_buffer, episode_values = [], []
                    episode_reward = episode_step_count = episode_inv_count = 0
                    d = False
    
                    # Initial state from the environment
                    if self.agentID==1:
                        self.env._reset(self.agentID)
                    self.synchronize() # synchronize starting time of the threads
                    validActions          = self.env._listNextValidActions(self.agentID)
                    s                     = self.env._observe(self.agentID)
                    blocking              = False
                    p=self.env.world.getPos(self.agentID)
                    on_goal               = self.env.world.goals[p[0],p[1]]==self.agentID
                    s                     = self.env._observe(self.agentID)
                    rnn_state             = self.local_AC.state_init
                    rnn_state0            = rnn_state
                    RewardNb = 0 
                    wrong_blocking  = 0
                    wrong_on_goal=0
    
                    if self.agentID==1:
                        global demon_probs
                        demon_probs[self.metaAgentID]=np.random.rand()
                    self.synchronize() # synchronize starting time of the threads
    
                    # reset swarm_reward (for tensorboard)
                    swarm_reward[self.metaAgentID] = 0
                    if episode_count<PRIMING_LENGTH or demon_probs[self.metaAgentID]<DEMONSTRATION_PROB:
                        #for the first PRIMING_LENGTH episodes, or with a certain probability
                        #don't train on the episode and instead observe a demonstration from M*
                        if self.workerID==1 and episode_count%100==0:
                            saver.save(sess, model_path+'/model-'+str(int(episode_count))+'.cptk')
                            curriculum.save_stage(model_path)
                        global rollouts
                        rollouts[self.metaAgentID]=None
                        if(self.agentID==1):
                            world=self.env.getObstacleMap()
                            start_positions=tuple(self.env.getPositions())
                            goals=tuple(self.env.getGoals())
                            try:
                                mstar_path=cpp_mstar.find_path(world,start_positions,goals,2,5)
                                rollouts[self.metaAgentID]=self.parse_path(mstar_path)
                            except OutOfTimeError:
                                #M* timed out 
                                print("timeout",episode_count)
                            except NoSolutionError:
                                print("nosol????",episode_count,start_positions)
                        self.synchronize()
                        if rollouts[self.metaAgentID] is not None:
                            agent_rollout = rollouts[self.metaAgentID][self.agentID-1]
                            if len(agent_rollout) > 0:
                                i_l=self.train(agent_rollout, sess, gamma, None, rnn_state0, imitation=True)
                            else:
                                i_l = 0.0
                            episode_count+=1./NUM_AGENTS
                            if self.agentID==1:
                                if pbar is not None:
                                    pbar.update(1)
                                    pbar.set_postfix({"imit_loss": f"{i_l:.4f}"})
                                summary = tf.Summary()
                                summary.value.add(tag='Losses/Imitation loss', simple_value=i_l)
                                global_summary.add_summary(summary, int(episode_count))
                                global_summary.flush()
                            continue
                        continue
                    saveGIF = False
                    if OUTPUT_GIFS and self.workerID == 1 and ((not TRAINING) or (episode_count >= self.nextGIF)):
                        saveGIF = True
                        self.nextGIF =episode_count + 64
                        GIF_episode = int(episode_count)
                        episode_frames = [ self.env._render(mode='rgb_array',screen_height=900,screen_width=900) ]
                        
                    while (not self.env.finished): # Give me something!
                        #Take an action using probabilities from policy network output.
                        a_dist,v,rnn_state,pred_blocking,pred_on_goal = sess.run([self.local_AC.policy,
                                                       self.local_AC.value,
                                                       self.local_AC.state_out,
                                                       self.local_AC.blocking,
                                                        self.local_AC.on_goal], 
                                             feed_dict={self.local_AC.inputs:[s[0]],
                                                        self.local_AC.goal_pos:[s[1]],
                                                        self.local_AC.state_in[0]:rnn_state[0],
                                                        self.local_AC.state_in[1]:rnn_state[1]})
    
                        if(not (np.argmax(a_dist.flatten()) in validActions)):
                            episode_inv_count += 1
                        train_valid = np.zeros(a_size)
                        train_valid[validActions] = 1
    
                        valid_dist = np.array([a_dist[0,validActions]])
                        valid_dist /= np.sum(valid_dist)
    
                        if TRAINING:
                            if (pred_blocking.flatten()[0] < 0.5) == blocking:
                                wrong_blocking += 1
                            if (pred_on_goal.flatten()[0] < 0.5) == on_goal:
                                wrong_on_goal += 1
                            a           = validActions[ np.random.choice(range(valid_dist.shape[1]),p=valid_dist.ravel()) ]
                            train_val   = 1.
                        else:
                            a         = np.argmax(a_dist.flatten())
                            if a not in validActions or not GREEDY:
                                a     = validActions[ np.random.choice(range(valid_dist.shape[1]),p=valid_dist.ravel()) ]
                            train_val = 1.
    
                        _, r, _, _, on_goal,blocking,_ = self.env._step((self.agentID, a),episode=episode_count)
    
                        self.synchronize() # synchronize threads
    
                        # Get common observation for all agents after all individual actions have been performed
                        s1           = self.env._observe(self.agentID)
                        validActions = self.env._listNextValidActions(self.agentID, a,episode=episode_count)
                        d            = self.env.finished
    
                        if saveGIF:
                            episode_frames.append(self.env._render(mode='rgb_array',screen_width=900,screen_height=900))
    
                        episode_buffer.append([s[0],a,r,s1,d,v[0,0],train_valid,pred_on_goal,int(on_goal),pred_blocking,int(blocking),s[1],train_val])
                        episode_values.append(v[0,0])
                        episode_reward += r
                        s = s1
                        total_steps += 1
                        episode_step_count += 1
    
                        if r>0:
                            RewardNb += 1
    
                        # If the episode hasn't ended, but the experience buffer is full, then we
                        # make an update step using that experience rollout.
                        if TRAINING and (len(episode_buffer) % EXPERIENCE_BUFFER_SIZE == 0 or d):
                            # Since we don't know what the true final return is, we "bootstrap" from our current value estimation.
                            if len(episode_buffer) >= EXPERIENCE_BUFFER_SIZE:
                                episode_buffers[i_buf] = episode_buffer[-EXPERIENCE_BUFFER_SIZE:]
                            else:
                                episode_buffers[i_buf] = episode_buffer[:]
    
                            if d:
                                s1Values[i_buf] = 0
                            else:
                                s1Values[i_buf] = sess.run(self.local_AC.value, 
                                     feed_dict={self.local_AC.inputs:np.array([s[0]])
                                                ,self.local_AC.goal_pos:[s[1]]
                                                ,self.local_AC.state_in[0]:rnn_state[0]
                                                ,self.local_AC.state_in[1]:rnn_state[1]})[0,0]
    
                            if (episode_count-EPISODE_START) < NUM_BUFFERS:
                                i_rand = np.random.randint(i_buf+1)
                            else:
                                i_rand = np.random.randint(NUM_BUFFERS)
                                tmp = np.array(episode_buffers[i_rand], dtype=object)
                                while tmp.shape[0] == 0:
                                    i_rand = np.random.randint(NUM_BUFFERS)
                                    tmp = np.array(episode_buffers[i_rand], dtype=object)
                            v_l,p_l,valid_l,e_l,b_l,og_l,g_n,v_n = self.train(episode_buffers[i_rand],sess,gamma,s1Values[i_rand],rnn_state0)
    
                            i_buf = (i_buf + 1) % NUM_BUFFERS
                            rnn_state0             = rnn_state
                            episode_buffers[i_buf] = []
    
                        self.synchronize() # synchronize threads
                        # sess.run(self.pull_global)
                        if episode_step_count >= max_episode_length or d:
                            break
    
                    episode_lengths[self.metaAgentID].append(episode_step_count)
                    episode_mean_values[self.metaAgentID].append(np.nanmean(episode_values))
                    episode_invalid_ops[self.metaAgentID].append(episode_inv_count)
                    episode_wrong_blocking[self.metaAgentID].append(wrong_blocking)
                    # Record success: d==True means all agents reached goals within step limit
                    episode_successes[self.metaAgentID].append(1 if d else 0)
    
                    # Periodically save gifs of episodes, model parameters, and summary statistics.
                    if episode_count % EXPERIENCE_BUFFER_SIZE == 0 and printQ:
                        print('                                                                                   ', end='\r')
                        print('{} Episode terminated ({},{})'.format(episode_count, self.agentID, RewardNb), end='\r')
    
                    swarm_reward[self.metaAgentID] += episode_reward
    
                    self.synchronize() # synchronize threads
    
                    episode_rewards[self.metaAgentID].append(swarm_reward[self.metaAgentID])
    
                    if not TRAINING:
                        mutex.acquire()
                        if episode_count < NUM_EXPS:
                            plan_durations[episode_count] = episode_step_count
                        if self.workerID == 1:
                            episode_count += 1
                            print('({}) Thread {}: {} steps, {:.2f} reward ({} invalids).'.format(episode_count, self.workerID, episode_step_count, episode_reward, episode_inv_count))
                        GIF_episode = int(episode_count)
                        mutex.release()
                    else:
                        episode_count+=1./NUM_AGENTS
    
                        if self.agentID==1 and pbar is not None:
                            pbar.update(1)
                            pbar.set_postfix({
                                "reward": f"{swarm_reward[self.metaAgentID]:.1f}",
                                "steps":  episode_step_count,
                                "stage":  curriculum.stage,
                            })
    
                        if episode_count % SUMMARY_WINDOW == 0:
                            if episode_count % 100 == 0:
                                print ('Saving Model', end='\n')
                                saver.save(sess, model_path+'/model-'+str(int(episode_count))+'.cptk')
                                curriculum.save_stage(model_path)
                                print ('Saved Model', end='\n')
                                gc.collect()
                            SL = SUMMARY_WINDOW * NUM_AGENTS
                            mean_reward = np.nanmean(episode_rewards[self.metaAgentID][-SL:])
                            mean_length = np.nanmean(episode_lengths[self.metaAgentID][-SL:])
                            mean_value = np.nanmean(episode_mean_values[self.metaAgentID][-SL:])
                            mean_invalid = np.nanmean(episode_invalid_ops[self.metaAgentID][-SL:])
                            mean_wrong_blocking = np.nanmean(episode_wrong_blocking[self.metaAgentID][-SL:])
                            current_learning_rate = sess.run(lr,feed_dict={global_step:episode_count})
    
                            summary = tf.Summary()
                            summary.value.add(tag='Perf/Learning Rate',simple_value=current_learning_rate)
                            summary.value.add(tag='Perf/Reward', simple_value=mean_reward)
                            summary.value.add(tag='Perf/Length', simple_value=mean_length)
                            summary.value.add(tag='Perf/Valid Rate', simple_value=(mean_length-mean_invalid)/mean_length)
                            summary.value.add(tag='Perf/Blocking Prediction Accuracy', simple_value=(mean_length-mean_wrong_blocking)/mean_length)
    
                            summary.value.add(tag='Losses/Value Loss', simple_value=v_l)
                            summary.value.add(tag='Losses/Policy Loss', simple_value=p_l)
                            summary.value.add(tag='Losses/Blocking Loss', simple_value=b_l)
                            summary.value.add(tag='Losses/On Goal Loss', simple_value=og_l)
                            summary.value.add(tag='Losses/Valid Loss', simple_value=valid_l)
                            summary.value.add(tag='Losses/Grad Norm', simple_value=g_n)
                            summary.value.add(tag='Losses/Var Norm', simple_value=v_n)
                            global_summary.add_summary(summary, int(episode_count))
                            global_summary.flush()
    
                            if printQ:
                                print('{} Tensorboard updated ({})'.format(episode_count, self.workerID), end='\r')
    
                            # ---- Curriculum evaluation (only meta-agent 0, agentID 1) ----
                            if self.agentID == 1 and self.metaAgentID == 0:
                                # Aggregate last CURRICULUM_EVAL_WINDOW episodes across all meta-agents
                                SL_C = CURRICULUM_EVAL_WINDOW * NUM_AGENTS
                                all_rewards  = sum([episode_rewards[ma][-SL_C:]  for ma in range(NUM_META_AGENTS)], [])
                                all_lengths  = sum([episode_lengths[ma][-SL_C:]  for ma in range(NUM_META_AGENTS)], [])
                                all_invalids = sum([episode_invalid_ops[ma][-SL_C:] for ma in range(NUM_META_AGENTS)], [])
                                all_success  = sum([episode_successes[ma][-SL_C:] for ma in range(NUM_META_AGENTS)], [])
    
                                if len(all_lengths) > 0:
                                    c_mean_reward  = float(np.nanmean(all_rewards))
                                    c_mean_length  = float(np.nanmean(all_lengths))
                                    c_mean_invalid = float(np.nanmean(all_invalids))
                                    c_valid_rate   = (c_mean_length - c_mean_invalid) / c_mean_length if c_mean_length > 0 else 0.0
                                    c_mean_success = float(np.nanmean(all_success))
    
                                    curriculum.maybe_evaluate(
                                        episode_count, c_mean_reward, c_mean_length,
                                        c_valid_rate, c_mean_success,
                                        max_episode_length, global_summary
                                    )
    
                    if saveGIF:
                        # Dump episode frames for external gif generation (otherwise, makes the jupyter kernel crash)
                        time_per_step = 0.1
                        images = np.array(episode_frames)
                        if TRAINING:
                            make_gif(images, '{}/episode_{:d}_{:d}_{:.1f}.gif'.format(gifs_path,GIF_episode,episode_step_count,swarm_reward[self.metaAgentID]))
                        else:
                            make_gif(images, '{}/episode_{:d}_{:d}.gif'.format(gifs_path,GIF_episode,episode_step_count), duration=len(images)*time_per_step,true_image=True,salience=False)
                    if SAVE_EPISODE_BUFFER:
                        with open('gifs3D/episode_{}.dat'.format(GIF_episode), 'wb') as file:
                            pickle.dump(episode_buffer, file)
        except (tf.errors.CancelledError, RuntimeError):
            pass  # Session closed; exit worker thread gracefully.


## Training

In [5]:
# Learning parameters
max_episode_length     = 256
episode_count          = 0
EPISODE_START          = episode_count
gamma                  = .95 # discount rate for advantage estimation and reward discounting
#moved network parameters to ACNet.py
EXPERIENCE_BUFFER_SIZE = 128
GRID_SIZE              = 10 #the size of the FOV grid to apply to each agent
DIAG_MVMT              = False # Diagonal movements allowed?
a_size                 = 5 + int(DIAG_MVMT)*4
SUMMARY_WINDOW         = 20
NUM_AGENTS             = 4  # agents per environment (policy controls this many agents)
NUM_THREADS            = 1  # target total worker threads across all environments
NUM_META_AGENTS        = max(1, NUM_THREADS // NUM_AGENTS)  # number of parallel environments
NUM_BUFFERS            = 1 # NO EXPERIENCE REPLAY int(NUM_THREADS / 2)
EPISODE_SAMPLES        = EXPERIENCE_BUFFER_SIZE # 64
LR_Q                   = 2.e-5 #8.e-5 / NUM_THREADS # default: 1e-5
ADAPT_LR               = True
ADAPT_COEFF            = 1.e-5 #the coefficient A in LR_Q/sqrt(A*steps+1) for calculating LR
MAX_EPISODES           = 0# 0 = unlimited; set a finite value so training auto-stops
load_model             = False
RESET_TRAINER          = True
model_path             = 'model_primal'
gifs_path              = 'gifs_primal'
train_path             = 'train_primal'
GLOBAL_NET_SCOPE       = 'global'

#Imitation options
PRIMING_LENGTH         = 400    # number of episodes at the beginning to train only on demonstrations
DEMONSTRATION_PROB     = 0.5  # probability of training on a demonstration per episode

# Curriculum options
CURRICULUM_EVAL_WINDOW = 200   # evaluate difficulty every N episodes
CURRICULUM_WARMUP      = 2000  # no evaluation before this episode count

# Simulation options
FULL_HELP              = False
OUTPUT_GIFS            = False
SAVE_EPISODE_BUFFER    = False

# Testing
TRAINING               = True
GREEDY                 = False
NUM_EXPS               = 100
MODEL_NUMBER           = 0

# Shared arrays for tensorboard
episode_rewards        = [ [] for _ in range(NUM_META_AGENTS) ]
episode_lengths        = [ [] for _ in range(NUM_META_AGENTS) ]
episode_mean_values    = [ [] for _ in range(NUM_META_AGENTS) ]
episode_invalid_ops    = [ [] for _ in range(NUM_META_AGENTS) ]
episode_wrong_blocking = [ [] for _ in range(NUM_META_AGENTS) ]
episode_successes      = [ [] for _ in range(NUM_META_AGENTS) ]  # 1=success, 0=timeout
rollouts               = [ None for _ in range(NUM_META_AGENTS)]
demon_probs=[np.random.rand() for _ in range(NUM_META_AGENTS)]
# episode_steps_on_goal  = [ [] for _ in range(NUM_META_AGENTS) ]
printQ                 = False # (for headless)
swarm_reward           = [0]*NUM_META_AGENTS


In [ ]:
tf.reset_default_graph()
print("Hello World")
if not os.path.exists(model_path):
    os.makedirs(model_path)
config = tf.ConfigProto(allow_soft_placement = True)
config.gpu_options.allow_growth = True  # allocate GPU memory on demand, not upfront
# Do NOT set per_process_gpu_memory_fraction — let allow_growth handle it.
# A hard cap causes "failed to launch on the GPU" when the graph needs more memory.
config.intra_op_parallelism_threads = 1
config.inter_op_parallelism_threads = 1

if not TRAINING:
    plan_durations = np.array([0 for _ in range(NUM_EXPS)])
    mutex = threading.Lock()
    gifs_path += '_tests'
    if SAVE_EPISODE_BUFFER and not os.path.exists('gifs3D'):
        os.makedirs('gifs3D')

#Create a directory to save episode playback gifs to
if not os.path.exists(gifs_path):
    os.makedirs(gifs_path)

print("Active Python threads before start:", threading.active_count())
if threading.active_count() > 32:
    raise RuntimeError("Too many leftover threads. Please restart the kernel and rerun from Cell 2.")

device_name = "/gpu:0" if USE_GPU else "/cpu:0"
print("Using device:", device_name)
print("Training with agents:", NUM_AGENTS)
print("Requested worker threads:", NUM_THREADS)
print("Parallel environments:", NUM_META_AGENTS)
print("Effective worker threads:", NUM_META_AGENTS * NUM_AGENTS)
if NUM_META_AGENTS * NUM_AGENTS != NUM_THREADS:
    print("Note: effective worker threads are quantized by NUM_AGENTS per environment.")

# ---- Curriculum scheduler ----
# Load saved stage BEFORE creating environments so envs are initialised at
# the correct difficulty from the very first episode.
curriculum = CurriculumScheduler(initial_stage=0)
if load_model:
    curriculum.load_stage(model_path)
print(f"Curriculum scheduler at Stage {curriculum.stage}: "
      f"size={curriculum.current_size}, density={curriculum.current_density}")

with tf.device(device_name):
    master_network = ACNet(GLOBAL_NET_SCOPE,a_size,None,False,GRID_SIZE,GLOBAL_NET_SCOPE) # Generate global network

    global_step = tf.placeholder(tf.float32)
    if ADAPT_LR:
        #computes LR_Q/sqrt(ADAPT_COEFF*steps+1)
        #we need the +1 so that lr at step 0 is defined
        lr=tf.divide(tf.constant(LR_Q),tf.sqrt(tf.add(1.,tf.multiply(tf.constant(ADAPT_COEFF),global_step))))
    else:
        lr=tf.constant(LR_Q)
    trainer = tf.train.AdamOptimizer(learning_rate=lr, use_locking=True)

    # Each environment requires one worker per agent for lock-step stepping.
    num_workers = NUM_AGENTS
    if not TRAINING:
        NUM_META_AGENTS = 1
    
    gameEnvs, workers, groupLocks = [], [], []
    n=1#counter of total number of agents (for naming)
    for ma in range(NUM_META_AGENTS):
        num_agents=NUM_AGENTS
        # Use curriculum's current size/density so continued training starts
        # at the correct stage rather than always at Stage 0.
        gameEnv = mapf_gym.MAPFEnv(num_agents=num_agents, DIAGONAL_MOVEMENT=DIAG_MVMT,
                                   SIZE=curriculum.current_size,
                                   observation_size=GRID_SIZE,
                                   PROB=curriculum.current_density, FULL_HELP=FULL_HELP)
        gameEnvs.append(gameEnv)

        # Create groupLock
        workerNames = ["worker_"+str(i) for i in range(n,n+num_workers)]
        groupLock = GroupLock.GroupLock([workerNames,workerNames])
        groupLocks.append(groupLock)

        # Create worker classes
        workersTmp = []
        for i in range(ma*num_workers+1,(ma+1)*num_workers+1):
            workersTmp.append(Worker(gameEnv,ma,n,a_size,groupLock))
            n+=1
        workers.append(workersTmp)

    # Register game environments with the curriculum scheduler so it can
    # update SIZE/PROB on all environments when a stage change occurs.
    curriculum.register_envs(gameEnvs)

    global_summary = tf.summary.FileWriter(train_path)
    saver = tf.train.Saver(max_to_keep=2)

    with tf.Session(config=config) as sess:
        sess.run(tf.global_variables_initializer())
        coord = tf.train.Coordinator()
        if load_model == True:
            print ('Loading Model...')
            if not TRAINING:
                with open(model_path+'/checkpoint', 'w') as file:
                    file.write('model_checkpoint_path: "model-{}.cptk"'.format(MODEL_NUMBER))
                    file.close()
            ckpt = tf.train.get_checkpoint_state(model_path)
            p=ckpt.model_checkpoint_path
            p=p[p.find('-')+1:]
            p=p[:p.find('.')]
            episode_count=int(p)
            saver.restore(sess,ckpt.model_checkpoint_path)
            print("episode_count set to ",episode_count)
            if RESET_TRAINER:
                trainer = tf.train.AdamOptimizer(learning_rate=lr, use_locking=True)

        # --- Pre-warm GPU: load cuDNN kernels in the main thread before worker threads start ---
        # cuDNN sub-libraries (~592MB) fail to mmap when lazy-loaded from child threads.
        if USE_GPU:
            print("Pre-warming GPU (loading cuDNN libraries)...")
            _warm_worker = workers[0][0]
            _warm_env = gameEnvs[0]
            _warm_env._reset(1)
            _warm_s = _warm_env._observe(1)
            _warm_rnn = _warm_worker.local_AC.state_init
            # Forward pass (loads cuDNN inference kernels)
            sess.run([_warm_worker.local_AC.policy, _warm_worker.local_AC.value,
                      _warm_worker.local_AC.state_out, _warm_worker.local_AC.blocking,
                      _warm_worker.local_AC.on_goal],
                     feed_dict={_warm_worker.local_AC.inputs: [_warm_s[0]],
                                _warm_worker.local_AC.goal_pos: [_warm_s[1]],
                                _warm_worker.local_AC.state_in[0]: _warm_rnn[0],
                                _warm_worker.local_AC.state_in[1]: _warm_rnn[1]})
            # Backward pass (loads cuDNN training kernels)
            _warm_ro = np.array([[_warm_s[0], _warm_s[1], 0]], dtype=object)
            sess.run([_warm_worker.local_AC.imitation_loss, _warm_worker.local_AC.apply_imitation_grads],
                     feed_dict={global_step: 0,
                                _warm_worker.local_AC.inputs: np.stack(_warm_ro[:, 0]),
                                _warm_worker.local_AC.goal_pos: np.stack(_warm_ro[:, 1]),
                                _warm_worker.local_AC.optimal_actions: np.stack(_warm_ro[:, 2]),
                                _warm_worker.local_AC.state_in[0]: _warm_rnn[0],
                                _warm_worker.local_AC.state_in[1]: _warm_rnn[1]})
            _warm_env._reset(1)  # reset env after warm-up
            print("GPU pre-warm complete.")
        # --- End pre-warm ---

        # Create progress bar
        pbar_total = MAX_EPISODES if MAX_EPISODES > 0 else None
        pbar = tqdm_bar(total=pbar_total, desc="Training", unit="ep", initial=int(episode_count))

        # Start the "work" process for each worker in a separate thread.
        worker_threads = []
        for ma in range(NUM_META_AGENTS):
            for worker in workers[ma]:
                groupLocks[ma].acquire(0,worker.name)
                worker_work = lambda w=worker: w.work(max_episode_length,gamma,sess,coord,saver,pbar if w.agentID==1 else None)
                print("Starting worker " + str(worker.workerID))
                t = threading.Thread(target=worker_work, daemon=True)
                t.start()
                worker_threads.append(t)

        # Coordinator.join can deadlock if a worker blocks on GroupLock at shutdown.
        # Use an explicit stop sequence once the target episode is reached.
        if TRAINING and MAX_EPISODES > 0:
            while episode_count < MAX_EPISODES and any(t.is_alive() for t in worker_threads):
                time.sleep(0.2)
            coord.request_stop()
            for gl in groupLocks:
                gl.releaseAll()
        else:
            # Unlimited training: keep session alive until manually interrupted.
            try:
                while any(t.is_alive() for t in worker_threads):
                    time.sleep(0.5)
            except KeyboardInterrupt:
                pass
            finally:
                coord.request_stop()
                for gl in groupLocks:
                    gl.releaseAll()

            t.join(timeout=5)

        pbar.close()
        print(f"\nTraining finished at episode {int(episode_count)}.")
        print(f"Final curriculum stage: {curriculum.stage} "
              f"(size={curriculum.current_size}, density={curriculum.current_density})")

if not TRAINING:
    print([np.mean(plan_durations), np.sqrt(np.var(plan_durations)), np.mean(np.asarray(plan_durations < max_episode_length, dtype=float))])


Hello World
Active Python threads before start: 8
Using device: /gpu:0
Training with agents: 4
Requested worker threads: 1
Parallel environments: 1
Effective worker threads: 4
Note: effective worker threads are quantized by NUM_AGENTS per environment.
Curriculum scheduler at Stage 0: size=(10, 10), density=(0.0, 0.1)
Instructions for updating:
Please use `keras.layers.RNN(cell)`, which is equivalent to this API
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor


/opt/conda/envs/primal/lib/python3.10/site-packages/tensorflow/python/keras/engine/base_layer_v1.py:1694: UserWarning: `layer.apply` is deprecated and will be removed in a future version. Please use `layer.__call__` method instead.
  warnings.warn('`layer.apply` is deprecated and '
/opt/conda/envs/primal/lib/python3.10/site-packages/tensorflow/python/keras/legacy_tf_layers/core.py:318: UserWarning: `tf.layers.flatten` is deprecated and will be removed in a future version. Please use `tf.keras.layers.Flatten` instead.
  warnings.warn('`tf.layers.flatten` is deprecated and '
/workspaces/PRIMAL/ACNet.py:105: UserWarning: `tf.nn.rnn_cell.BasicLSTMCell` is deprecated and will be removed in a future version. This class is equivalent as `tf.keras.layers.LSTMCell`, and will be replaced by that in Tensorflow 2.0.
  lstm_cell = tf.nn.rnn_cell.BasicLSTMCell(RNN_SIZE,state_is_tuple=True)


Hello World... From  global
Hello World... From  worker_1
Hello World... From  worker_2
Hello World... From  worker_3
Hello World... From  worker_4
Pre-warming GPU (loading cuDNN libraries)...
GPU pre-warm complete.


Training: 0ep [00:00, ?ep/s]

Starting worker 1
Starting worker 2
Starting worker 3
Starting worker 4


...

Training: 41600ep [13:15:32,  1.71s/ep, reward=-11.9, steps=15, stage=5]  

Saving Model
Saving Model
Saved Model
Saved Model


Training: 41700ep [13:18:02,  2.74s/ep, reward=-95.9, steps=256, stage=5] 

Saving Model
Saving Model
Saved Model
Saved Model


Training: 41818ep [13:20:54,  1.81s/ep, reward=-217.9, steps=256, stage=5]